In [ ]:
# Import necessary libraries
from kfp.v2 import dsl
from kfp.v2.dsl import component
from kfp.v2 import compiler
from google.cloud import aiplatform

# Define a simple component
@component(base_image="python:3.9", packages_to_install=["pandas==2.2.0", "numpy"])
def hello_world_component():
    import pandas as pd
    print("Hello, World!")
# Define the pipeline
@dsl.pipeline(
    name="hello-world-pipeline",
    description="A simple Hello World pipeline"
)
def hello_world_pipeline():
    hello_world_component()

# Compile the pipeline
if __name__ == "__main__":
    compiler.Compiler().compile(
        pipeline_func=hello_world_pipeline,
        package_path="hello_world_pipeline.json"
    )

# Initialize the Vertex AI client
aiplatform.init(
    project="anbc-dev-hcm-cm-de",  # Replace with your GCP project ID
    location="us-east4",           # Replace with your region
)

# Define the pipeline job
pipeline_job = aiplatform.PipelineJob(
    display_name="simple-hello-pipeline",  # Name of the pipeline
    template_path="hello_world_pipeline.json",  # Path to the compiled pipeline JSON file
    pipeline_root="gs://hcm-cm-de-code-hcb-dev/vertex-test/",  # Replace with your GCS bucket path
    parameter_values={},  # Pass parameters to the pipeline
    encryption_spec_key_name="projects/cvs-key-vault-nonprod/locations/us-east4/keyRings/gkr-nonprod-us-east4/cryptoKeys/gk-anbc-dev-hcm-cm-de-us-east4"  # Specify the CMEK
)

# Submit the pipeline job
pipeline_job.run(
    service_account="gchcb-hcm-cm-de-ontpd@anbc-dev-hcm-cm-de.iam.gserviceaccount.com",
    sync=True
)

Creating PipelineJob
PipelineJob created. Resource name: projects/46378383599/locations/us-east4/pipelineJobs/hello-world-pipeline-20250926150245
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/46378383599/locations/us-east4/pipelineJobs/hello-world-pipeline-20250926150245')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-east4/pipelines/runs/hello-world-pipeline-20250926150245?project=46378383599
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/hello-world-pipeline-20250926150245 current state:
PipelineState.PIPELINE_STATE_RUNNING
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/hello-world-pipeline-20250926150245 current state:
PipelineState.PIPELINE_STATE_RUNNING
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/hello-world-pipeline-20250926150245 current state:
PipelineState.PIPELINE_STATE_RUNNING
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/

In [1]:
import utils.gcp_handling
import utils.components as components
import os
# Import necessary libraries
from kfp.v2 import dsl
from kfp.v2.dsl import component
from kfp.v2 import compiler
from google.cloud import aiplatform

c:\Program Files\Python\313\Lib\site-packages\kfp\dsl\component_decorator.py:126: FutureWarning: The default base_image used by the @dsl.component decorator will switch from 'python:3.9' to 'python:3.10' on Oct 1, 2025. To ensure your existing components work with versions of the KFP SDK released after that date, you should provide an explicit base_image argument and ensure your component works as intended on Python 3.10.
  return component_factory.create_component_from_func(


In [2]:
user_constants = {
    "EMAIL": "palmere1_aetna_com", # e.g., john.doe@cvshealth.com
    "COSTCENTER": "13070", # Insert your costcenter here
    "TENANT": "hcm-cm-de", # e.g., mleng-platform
    "USE_COMPUTE_PROJECT": True, # e.g., True or False
    "OWNER": "palmere1_aetna_com",
    "COMPUTE_PROJECT": "anbc-dev-hcm-cm-de",
    "PROJECT": "anbc-dev-hcm-cm-de",
    'LABELS': {'owner': 'palmere1_aetna_com',
            'costcenter': '13070',
            'tenant': 'hcm-cm-de',
            'self_serve': 'true',
            'lob': 'hcb',
            'pipeline_type': 'training_prediction'},
    # 'hcb' or 'pss' or 'ent'
    "LOB": "hcb",
    # save_location -> may be helpful for tracking artifacts / model registration etc
    # 'MANUAL_SAVE_LOCATION': 'gs://',
    # name of this repo
    # If defined, overwrites system-derived REPO_NAME value
    "MODEL_DESCRIPTION": "maternity_09_25", # Description for model when registered with ML Platform
    "PIPELINE_TYPE": "training_prediction", # Supported values: training, prediction, evaluation, training_prediction, other
    
    # SQL Variables for BigQuery queries
    "GCP_PROJECT": "anbc-hcb-dev",
    "GCP_DB": "cm_medicaid_hcb_dev", 
    "PREFIX": "a534354_mlops_test",
    "DEFAULT_EXP": "INTERVAL 2 DAY",
    "ST": "anbc-hcb-dev.cm_medicaid_hcb_dev.a534354_mat_v2_st",
    "SDOH_YEAR":'2023'
}

In [3]:
constants = user_constants
os.environ["OWNER"] = constants["EMAIL"]
os.environ["COSTCENTER"] = constants["COSTCENTER"]

In [4]:
def process_sql_file(sql_file_path, constants, base_path="../sql_queries/"):
    import os
    import re
    
    # Construct full path
    full_path = os.path.join(base_path, sql_file_path)
    
    # Read the SQL file content
    with open(full_path, 'r') as file:
        sql_content = file.read()
    
    # Find all variables in the SQL file
    variables_in_sql = re.findall(r'\{([^}]+)\}', sql_content)
    
    # Create a clean substitution dictionary from constants
    substitution_dict = {}
    
    for key, value in constants.items():
        # Skip nested dictionaries and non-string values that aren't useful for SQL
        if isinstance(value, dict):
            continue  # Skip LABELS and other nested objects
        elif isinstance(value, bool):
            substitution_dict[key] = str(value).upper()  # Convert True/False to TRUE/FALSE
        else:
            substitution_dict[key] = str(value)  # Convert everything to string
    
    # Add additional mappings for common SQL variable patterns
    additional_mappings = {
        'COST_CENTER': constants.get('COSTCENTER', ''),  # Map COSTCENTER to COST_CENTER
    }
    
    # Merge additional mappings
    substitution_dict.update(additional_mappings)
    
    # Check which variables can be substituted
    missing_variables = []
    available_substitutions = {}
    
    for var_name in variables_in_sql:
        if var_name in substitution_dict:
            available_substitutions[var_name] = substitution_dict[var_name]
        else:
            missing_variables.append(var_name)
    
    # Warn about missing variables but don't fail
    if missing_variables:
        print(f"Warning: Variables not found in constants: {missing_variables}")
        print(f"Available substitutions: {list(substitution_dict.keys())}")
        print(f"Variables in SQL: {variables_in_sql}")
    
    # Substitute variables using **kwargs
    try:
        sql_query = sql_content.format(**available_substitutions)
    except KeyError as e:
        print(f"Error: Missing variable {e} in SQL file {sql_file_path}")
        print(f"Available substitutions: {list(available_substitutions.keys())}")
        raise
    
    return sql_query

In [12]:
print(process_sql_file('003_membership_cross_walk.sql', constants, base_path="../custom_dev_data_pull/"))

-----------------------------
--- All IDs cross-linking ---
-----------------------------
--take asdb IDs from target population (table a)
--cross-link to insights for iodb, QNXT, and edw individual_id (table b)
--cross-link to edw individual_id -> member_id so we can find all claims for a given member in the edw (table c)
--cross-link to komodo mdcd reidentification file (table d)
--cross-link to komodo non-mdcd reidentificaiton file (table e)
CREATE OR REPLACE TABLE `anbc-hcb-dev.cm_medicaid_hcb_dev.a534354_mlops_test_id_crosswalk`
OPTIONS (labels = [("owner", "palmere1_aetna_com"),("cost_center", "13070")]
         , expiration_timestamp = TIMESTAMP_ADD(CURRENT_TIMESTAMP(), INTERVAL 2 DAY))
AS
WITH d AS (
    SELECT DISTINCT 
        medicaid_id
        , upk_token_2
        , individual_id 
    FROM 
        `anbc-hcb-prod.eds_srcapp_komodombr_share_hcb_prod.enriched_active_aetna_beneficiary` 
    WHERE 1 = 1
        AND CAST(dt_field AS DATE) >= "2022-05-01"
)
SELECT DISTINCT
    

In [7]:
from google.cloud import aiplatform
import os

# Define the pipeline
@dsl.pipeline(
    name="bigquery-pipeline",
    description="Pipeline to perform BigQuery operations in sequential order"
)
def bigquery_pipeline(
    project_id: str = "anbc-dev-hcm-cm-de",
    region: str = "us-east4",
    cmek_key: str = "projects/cvs-key-vault-nonprod/locations/us-east4/keyRings/gkr-nonprod-us-east4/cryptoKeys/gk-anbc-dev-hcm-cm-de-us-east4"
):
    
    # This is the body of the pipeline
    # Use .after(component) to set flow of individual components
    # Use .after(*[component1, component2, componentN]) for setting after multiple components

    #################################
    # SQL Pipeline - Following the same order as 000_run.sh
    #################################

    # # Step 1: Medical Claims Year 1
    # sql_query_1 = process_sql_file('002_Med_Claims_yr1.sql', constants)
    # query_1_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_1,
    #     display_name="002_Med_Claims_yr1"
    # )
    
    # # Step 2: Medical Claims Year 2
    # sql_query_2 = process_sql_file('002_Med_Claims_yr2.sql', constants)
    # query_2_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_2,
    #     display_name="002_Med_Claims_yr2"
    # ).after(query_1_bq)
    
    # # Step 3: Cost and Utilization Year 1
    # sql_query_3 = process_sql_file('003_Cost_and_Utilization_yr1.sql', constants)
    # query_3_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_3,
    #     display_name="003_Cost_and_Utilization_yr1"
    # ).after(query_2_bq)
    
    # # Step 4: Cost and Utilization Year 2
    # sql_query_4 = process_sql_file('003_Cost_and_Utilization_yr2.sql', constants)
    # query_4_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_4,
    #     display_name="003_Cost_and_Utilization_yr2"
    # ).after(query_3_bq)
    
    # # Step 5: Conditions
    # sql_query_5 = process_sql_file('004_Conditions.sql', constants)
    # query_5_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_5,
    #     display_name="004_Conditions"
    # ).after(query_4_bq)
    
    # # Step 6: Rx Claims Year 1
    # sql_query_6 = process_sql_file('006_Rx_Claims_yr1.sql', constants)
    # query_6_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_6,
    #     display_name="006_Rx_Claims_yr1"
    # ).after(query_5_bq)
    
    # # Step 7: Rx Claims Year 2
    # sql_query_7 = process_sql_file('006_Rx_Claims_yr2.sql', constants)
    # query_7_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_7,
    #     display_name="006_Rx_Claims_yr2"
    # ).after(query_6_bq)
    
    # # Step 8: Demographics
    # sql_query_8 = process_sql_file('007_Demographics.sql', constants)
    # query_8_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_8,
    #     display_name="007_Demographics"
    # ).after(query_7_bq)
    
    # # Step 9: GeoID
    # sql_query_9 = process_sql_file('008_GeoID.sql', constants)
    # query_9_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_9,
    #     display_name="008_GeoID"
    # ).after(query_8_bq)
    
    # # Step 10: ACS Data
    # sql_query_10 = process_sql_file('009_ACS.sql', constants)
    # query_10_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_10,
    #     display_name="009_ACS"
    # ).after(query_9_bq)
    
    # # Step 11: Preventative Care
    # sql_query_11 = process_sql_file('010_preventative.sql', constants)
    # query_11_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_11,
    #     display_name="010_preventative"
    # ).after(query_10_bq)
    
    # # Step 12: CSDI Risk
    # sql_query_12 = process_sql_file('011_CSDI_risk.sql', constants)
    # query_12_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_12,
    #     display_name="011_CSDI_risk"
    # ).after(query_11_bq)
    
    # # Step 13: Non-Embedding Feature Beast (Final aggregation)
    # sql_query_13 = process_sql_file('013_non_embedding_feature_beast.sql', constants)
    # query_13_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_13,
    #     display_name="013_non_embedding_feature_beast"
    # ).after(query_12_bq)

    # sql_query_14 = process_sql_file('001_asdb_cpt_rev_icd.sql', constants, base_path="../custom_dev_data_pull/")
    # query_14_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_14,
    #     display_name="001_asdb_cpt_rev_icd"
    # ).after(query_13_bq)

    sql_query_15 = process_sql_file('002_asdb_cohort_id.sql', constants, base_path="../custom_dev_data_pull/")
    query_15_bq = components.query_bigquery_component(
        constants=constants,
        query=sql_query_15,
        display_name="002_asdb_cohort_id"
    )#.after(query_14_bq) # remove the .after() if we want to resume testing from here without caching.
        
    # sql_query_16 = process_sql_file('003_membership_cross_walk.sql', constants, base_path="../custom_dev_data_pull/")
    # query_16_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_16,
    #     display_name="003_membership_cross_walk"
    # ).after(query_15_bq)
    
    
    # sql_query_17 = process_sql_file('004_asdb_cpt_rev_icd.sql', constants, base_path="../custom_dev_data_pull/")
    # query_17_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_17,
    #     display_name="004_asdb_cpt_rev_icd"
    # ).after(query_16_bq)
    
    
    # sql_query_18 = process_sql_file('005_asdb_labs.sql', constants, base_path="../custom_dev_data_pull/")
    # query_18_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_18,
    #     display_name="005_asdb_labs"
    # ).after(query_17_bq)
    
    
    # sql_query_19 = process_sql_file('006_edw_cpt_rev_icd.sql', constants, base_path="../custom_dev_data_pull/")
    # query_19_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_19,
    #     display_name="006_edw_cpt_rev_icd"
    # ).after(query_18_bq)
    
    
    # sql_query_20 = process_sql_file('007_edw_labs.sql', constants, base_path="../custom_dev_data_pull/")
    # query_20_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_20,
    #     display_name="007_edw_labs"
    # ).after(query_19_bq)
    
    
    # sql_query_21 = process_sql_file('008_komodo_cpt_rev_icd.sql', constants, base_path="../custom_dev_data_pull/")
    # query_21_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_21,
    #     display_name="008_komodo_cpt_rev_icd"
    # ).after(query_20_bq)
    
    # sql_query_22 = process_sql_file('009_custom_predictors_joined.sql', constants, base_path="../custom_dev_data_pull/")
    # query_22_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_22,
    #     display_name="009_custom_predictors_joined"
    # ).after(query_21_bq)
    
    # sql_query_23 = process_sql_file('010_labs_joined.sql', constants, base_path="../custom_dev_data_pull/")
    # query_23_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_23,
    #     display_name="010_labs_joined"
    # ).after(query_22_bq)
    
    # sql_query_24 = process_sql_file('011_join_cust_pipeline_embed_features.sql', constants, base_path="../custom_dev_data_pull/")
    # query_24_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_24,
    #     display_name="011_join_cust_pipeline_embed_features"
    # ).after(query_23_bq)

    # sql_query_25 = process_sql_file('maternity_start.sql', constants, base_path="../custom_dev_data_pull/")
    # query_25_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_25,
    #     display_name="maternity_start"
    # ).after(query_24_bq)

    # sql_query_27 = process_sql_file('maternity_start2.sql', constants, base_path="../custom_dev_data_pull/")
    # query_27_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_27,
    #     display_name="maternity_start2"
    # ).after(query_25_bq)

    # sql_query_26 = process_sql_file('all_deliveries_to_20241014.sql', constants, base_path="../custom_dev_data_pull/")
    # query_26_bq = components.query_bigquery_component(
    #     constants=constants,
    #     query=sql_query_26,
    #     display_name="all_deliveries_to_20241014"
    # ).after(query_27_bq)

# Compile the pipeline
if __name__ == "__main__":
    compiler.Compiler().compile(
        pipeline_func=bigquery_pipeline,
        package_path="bigquery_pipeline.json"
    )

    # Initialize Vertex AI client
    aiplatform.init(
        project="anbc-dev-hcm-cm-de",  # Replace with your GCP project ID
        location="us-east4",           # Replace with your region
    )

    # Define the pipeline job
    pipeline_job = aiplatform.PipelineJob(
        display_name="bigquery-pipeline",
        template_path="bigquery_pipeline.json",
        pipeline_root="gs://hcm-cm-de-code-hcb-dev/vertex-test/",  # Replace with your GCS bucket path
        encryption_spec_key_name="projects/cvs-key-vault-nonprod/locations/us-east4/keyRings/gkr-nonprod-us-east4/cryptoKeys/gk-anbc-dev-hcm-cm-de-us-east4",  # Specify the CMEK
        parameter_values={
            "project_id": "anbc-dev-hcm-cm-de",
            "region": "us-east4",
            "cmek_key": "projects/cvs-key-vault-nonprod/locations/us-east4/keyRings/gkr-nonprod-us-east4/cryptoKeys/gk-anbc-dev-hcm-cm-de-us-east4"
        },
        labels={
        "owner": "sahil_gadge_aetna_com",
        "pipeline_type": "bigquery",
        "lob": "hcb",
        "costcenter": "13070",
        "tenant": "hcm-cm-de",
        "self_serve": "true"
    }
    )

    # Submit the pipeline job
    pipeline_job.run(
        service_account="gchcb-hcm-cm-de-ontpd@anbc-dev-hcm-cm-de.iam.gserviceaccount.com",
        sync=True
    )

Creating PipelineJob
PipelineJob created. Resource name: projects/46378383599/locations/us-east4/pipelineJobs/bigquery-pipeline-20251001000545
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/46378383599/locations/us-east4/pipelineJobs/bigquery-pipeline-20251001000545')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-east4/pipelines/runs/bigquery-pipeline-20251001000545?project=46378383599
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/bigquery-pipeline-20251001000545 current state:
PipelineState.PIPELINE_STATE_RUNNING
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/bigquery-pipeline-20251001000545 current state:
PipelineState.PIPELINE_STATE_RUNNING
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/bigquery-pipeline-20251001000545 current state:
PipelineState.PIPELINE_STATE_RUNNING
PipelineJob projects/46378383599/locations/us-east4/pipelineJobs/bigquery-pipeline-

RuntimeError: Job failed with:
code: 9
message: " The DAG failed because some tasks failed. The failed tasks are: [bigquery-query-job-27].; Job (project_id = anbc-dev-hcm-cm-de, job_id = 5148635820386680832) is failed due to the above error.; Failed to handle the job: {project_number = 46378383599, job_id = 5148635820386680832}"


In [8]:
process_sql_file('001_asdb_cpt_rev_icd.sql', constants, base_path="../custom_dev_data_pull/")

'-----------------------------------------------------------------------------------------\n--- Get all ICD and CPT codes so we can find relevant conditions and gestational ages ---\n-----------------------------------------------------------------------------------------\nCREATE OR REPLACE TABLE `anbc-hcb-dev.cm_medicaid_hcb_dev.a974930_sahil_test_all_icd_20240926`\nOPTIONS (labels = [("owner", "sahil_gadge_aetna_com"),("cost_center", "13070")]\n         , expiration_timestamp = TIMESTAMP_ADD(CURRENT_TIMESTAMP(), INTERVAL 2 DAY))\nAS\nWITH icd AS (\n    SELECT\n        clm.asdb_plan_key\n        , clm.claimid\n        , CASE WHEN LENGTH(TRIM(codeid)) = 3 THEN TRIM(UPPER(codeid))\n            WHEN SUBSTR(TRIM(UPPER(codeid)), 4, 1) = "." THEN TRIM(UPPER(codeid))\n            ELSE CONCAT(SUBSTR(TRIM(UPPER(codeid)), 1, 3), ".", SUBSTR(TRIM(codeid), 5, 8)) END AS codeid\n        , grp.ICD9_DX_GROUP_NBR AS icd_group\n    FROM\n        `edp-prod-hcbstorage.edp_hcb_mdcd_core_srcv.ASDB_ASDB_CL